In [1]:
  | symbol | VARCHAR |
  | timeframe | VARCHAR |
  | time | TIMESTAMP WITH TIME ZONE |
  | open | DOUBLE |
  | high | DOUBLE |
  | low | DOUBLE |
  | close | DOUBLE |
  | tick_volume | BIGINT |
  | spread | INTEGER |
  | real_volume | BIGINT |
  | created_at | TIMESTAMP |

SyntaxError: invalid syntax (2285982319.py, line 1)

In [4]:
# 1. Datenimport duckdb
from pathlib import Path
import duckdb
import pandas as pd

DB_PATH = Path(r"F:\Python\PyTrader\data\market_data.duckdb")
con = duckdb.connect(database=str(DB_PATH), read_only=True)


def load_candles(
    symbol: str, timeframe: str = "M5", limit: int = 1000
) -> pd.DataFrame:
    query = f"""
        SELECT "time", open, high, low, close, tick_volume AS volume
        FROM (
            SELECT 
                "time",
                open, 
                high, 
                low, 
                close, 
                tick_volume
            FROM ohlcv_bars 
            WHERE LOWER(symbol) = LOWER('{symbol}') 
              AND LOWER(timeframe) = LOWER('{timeframe}')
              AND "time" IS NOT NULL 
              AND open IS NOT NULL AND high IS NOT NULL 
              AND low IS NOT NULL AND close IS NOT NULL
            ORDER BY "time" DESC
            LIMIT {limit}
        ) sub
        ORDER BY "time" ASC;
    """
    with duckdb.connect(database=str(DB_PATH), read_only=True) as con:
        df = con.execute(query).df()
    
    # 1. Zeitzone abstreifen
    df["time"] = df["time"].dt.tz_localize(None)
    
    # 2. Browser-Offset (UTC+2 Sommerzeit) ausgleichen, damit die Wanduhrzeit 1:1 stimmt
    df["time"] = df["time"] - pd.Timedelta(hours=2)
    
    # 3. Cast auf datetime64[ns]
    df["time"] = df["time"].astype("datetime64[ns]")
    
    # Float-Werte absichern
    for col in ["open", "high", "low", "close", "volume"]:
        df[col] = df[col].astype("float64")

    return df


df = load_candles(symbol="SILVER", timeframe="M5", limit=1000)

In [5]:
# 2. Indikator und Signale
import numpy as np
import pandas as pd


def add_indicators_and_signals(data: pd.DataFrame) -> pd.DataFrame:
    df_calc = data.copy()

    # EMAs berechnen
    df_calc["ema_fast"] = df_calc["close"].ewm(span=9, adjust=False).mean()
    df_calc["ema_slow"] = df_calc["close"].ewm(span=21, adjust=False).mean()

    # Crossovers
    prev_fast = df_calc["ema_fast"].shift(1)
    prev_slow = df_calc["ema_slow"].shift(1)

    buy_cond = (df_calc["ema_fast"] > df_calc["ema_slow"]) & (
        prev_fast <= prev_slow
    )
    sell_cond = (df_calc["ema_fast"] < df_calc["ema_slow"]) & (
        prev_fast >= prev_slow
    )

    df_calc["signal"] = np.select([buy_cond, sell_cond], [1, -1], default=0)
    return df_calc


df_signals = add_indicators_and_signals(df)

In [6]:
# 3. Rendering
import pandas as pd
from lightweight_charts import JupyterChart


def plot_chart(df: pd.DataFrame):
    data = df.copy()

    chart = JupyterChart(width=1000, height=600)
    chart.layout(background_color="#131722", text_color="#d1d4dc")
    chart.candle_style(
        up_color="#26a69a",
        down_color="#ef5350",
        border_up_color="#26a69a",
        border_down_color="#ef5350",
        wick_up_color="#26a69a",
        wick_down_color="#ef5350",
    )

    # 1. OHLC-Kerzen setzen (time ist datetime64[ns])
    chart.set(data[["time", "open", "high", "low", "close"]])

    # 2. Indikatoren setzen
    line_fast = chart.create_line(name="EMA 9", color="#2962FF", width=2)
    line_fast.set(
        data[["time", "ema_fast"]].rename(columns={"ema_fast": "EMA 9"})
    )

    line_slow = chart.create_line(name="EMA 21", color="#FF6D00", width=2)
    line_slow.set(
        data[["time", "ema_slow"]].rename(columns={"ema_slow": "EMA 21"})
    )

    # 3. Marker setzen
    signals = data[data["signal"] != 0]
    for row in signals.itertuples(index=False):
        chart.marker(
            time=row.time,
            position="below" if row.signal == 1 else "above",
            shape="arrow_up" if row.signal == 1 else "arrow_down",
            color="#26a69a" if row.signal == 1 else "#ef5350",
            text="BUY" if row.signal == 1 else "SELL",
        )

    chart.load()


plot_chart(df_signals)

In [ ]:
# Export CSV
def export_signals(df_with_signals: pd.DataFrame, output_format="parquet"):
    # Nur echte Signale filtern
    signals_only = df_with_signals[df_with_signals["signal"] != 0][
        ["time", "close", "signal"]
    ]

    if output_format == "parquet":
        signals_only.to_parquet("exported_signals.parquet")
    elif output_format == "csv":
        signals_only.to_csv("exported_signals.csv", index=False)
    elif output_format == "duckdb":
        # Zurück in eine Export-DuckDB schreiben
        export_con = duckdb.connect("export_database.duckdb")
        export_con.execute(
            "CREATE OR REPLACE TABLE signals AS SELECT * FROM signals_only"
        )

    print(f"{len(signals_only)} Signale erfolgreich exportiert.")


export_signals(df_signals, output_format="parquet")

In [1]:
import duckdb
from pathlib import Path
import pandas as pd

# Daten laden
DB_PATH = Path(r"F:\Python\PyTrader\data\market_data.duckdb")
con = duckdb.connect(database=str(DB_PATH), read_only=True)

query = """
    SELECT 
        EXTRACT('epoch' FROM time)::BIGINT as time_epoch,
        open, high, low, close, tick_volume 
    FROM ohlcv_bars 
    WHERE symbol = 'SILVER' AND timeframe = 'M5'
    ORDER BY time ASC
    LIMIT 1000
"""
df = con.execute(query).df()
con.close()

# Daten prüfen
print("Erste 5 Zeilen:")
print(df.head())
print("\nDatentypen:")
print(df.dtypes)
print("\nZeitbereich:")
print(f"Min time_epoch: {df['time_epoch'].min()}")
print(f"Max time_epoch: {df['time_epoch'].max()}")
print(f"Anzahl Zeilen: {len(df)}")

Erste 5 Zeilen:
   time_epoch    open    high     low   close  tick_volume
0  1370390400  22.513  22.761  22.340  22.518        98547
1  1370476800  22.503  22.889  22.308  22.533        74864
2  1370563200  22.625  22.799  21.569  21.675       108311
3  1370822400  21.590  22.079  21.349  21.898        63576
4  1370908800  21.909  22.008  21.466  21.628        70306

Datentypen:
time_epoch       int64
open           float64
high           float64
low            float64
close          float64
tick_volume      int64
dtype: object

Zeitbereich:
Min time_epoch: 1370390400
Max time_epoch: 1488466800
Anzahl Zeilen: 1000


In [ ]:
# Nach Kernel-Neustart - alternative Methode
import subprocess
import tempfile
import pandas as pd
import os

# Testdaten
test_df = pd.DataFrame({
    'time': [1609459200000, 1609462800000, 1609466400000, 1609470000000],
    'open': [100, 101, 102, 103],
    'high': [102, 103, 104, 105],
    'low': [98, 99, 100, 101],
    'close': [101, 102, 103, 104]
})

# In temporäre CSV speichern
tmp_file = tempfile.NamedTemporaryFile(suffix='.csv', delete=False)
test_df.to_csv(tmp_file.name, index=False)

# Python-Skript schreiben, das den Chart ausführt
script = f'''
import pandas as pd
from lightweight_charts import Chart

df = pd.read_csv(r"{tmp_file.name}")
chart = Chart(width=800, height=400)
chart.set(df)
chart.show(block=True)
'''

# In temp-Datei schreiben
script_file = tempfile.NamedTemporaryFile(suffix='.py', delete=False)
script_file.write(script.encode())
script_file.close()

# Als Subprozess ausführen
subprocess.run(['python', script_file.name])

In [28]:
print("--- DATATYPES & SHAPE ---")
print(df.dtypes)
print(f"Shape: {df.shape}")

print("\n--- ERSTE 3 ZEILEN (ROHWERTE) ---")
print(df[["time", "open", "high", "low", "close"]].head(3))

print("\n--- TYP DES ERSTEN ZEITEINTRAGS ---")
sample_t = df["time"].iloc[0]
print(f"Wert: {sample_t!r} | Typ: {type(sample_t)}")

--- DATATYPES & SHAPE ---
time      datetime64[us, Europe/Budapest]
open                              float64
high                              float64
low                               float64
close                             float64
volume                            float64
dtype: object
Shape: (1000, 6)

--- ERSTE 3 ZEILEN (ROHWERTE) ---
                       time    open    high     low   close
0 2026-08-13 22:40:00+02:00  64.729  64.729  64.626  64.643
1 2026-08-13 22:45:00+02:00  64.651  64.663  64.607  64.610
2 2026-08-13 22:50:00+02:00  64.613  64.622  64.491  64.538

--- TYP DES ERSTEN ZEITEINTRAGS ---
Wert: Timestamp('2026-08-13 22:40:00+0200', tz='Europe/Budapest') | Typ: <class 'pandas.Timestamp'>
